# Data Preprocessing

The tasks behind this section are the following:


1.   Clean the WBL dataset including removing World Bank aggregate rows, kept only real economies.


**Sources:**
1.  Little & Rubin (2019) Ch. 1-2
Van Buuren (2018) Ch. 1
2.   Huyen (2022) Ch. 4
3.  Maharaj et al. (2019)
4. Hyland, Djankov & Goldberg (2020, AER: Insights) — feature-specific
fallback years, non-1971 baselines reported in data appendix.


**Saves:**

1. data/processed/wbl_panel_final.csv (189 economies — analysis ready)
2. outputs/figures/observed_years_distribution.png
3. outputs/figures/missingness_heatmap.png
4. reports/DESIGN_DECISIONS.md (appended)



In [1]:
# Change this path to match yours
BASE = '/content/drive/MyDrive/wbl_project'

In [2]:
# uncomment both lines if you want to connect this notebook to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Download all library
import os
import wbgapi as wb
import pandas as pd


# Install dependencies
# wbgapi: official World Bank Python client for Open Data API
# Documented at: https://pypi.org/project/wbgapi
!pip install wbgapi -q
!pip install missingno -q

# Uncomment if you want to check that the Path exists
#print(f'Exists: {os.path.exists(BASE)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Create folder structure in Google Collab
folders = [
    f'{BASE}/data/raw',
    f'{BASE}/data/processed',
    f'{BASE}/data/external',
    f'{BASE}/data/features',
    f'{BASE}/notebooks',
    f'{BASE}/outputs/figures',
    f'{BASE}/outputs/tables',
    f'{BASE}/reports/eda',
    f'{BASE}/reports/monitoring',
    f'{BASE}/models',
    f'{BASE}/tests',
    f'{BASE}/src',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'{folder}')

# .gitkeep ensures empty folders appear on GitHub
# Git does not track empty folders by default
for folder in [f'{BASE}/notebooks', f'{BASE}/src']:
    with open(f'{folder}/.gitkeep', 'w') as f:
        f.write('')

print('\n✅ Folder structure ready')

/content/drive/MyDrive/wbl_project/data/raw
/content/drive/MyDrive/wbl_project/data/processed
/content/drive/MyDrive/wbl_project/data/external
/content/drive/MyDrive/wbl_project/data/features
/content/drive/MyDrive/wbl_project/notebooks
/content/drive/MyDrive/wbl_project/outputs/figures
/content/drive/MyDrive/wbl_project/outputs/tables
/content/drive/MyDrive/wbl_project/reports/eda
/content/drive/MyDrive/wbl_project/reports/monitoring
/content/drive/MyDrive/wbl_project/models
/content/drive/MyDrive/wbl_project/tests
/content/drive/MyDrive/wbl_project/src

✅ Folder structure ready


In [5]:
# Create src/utils.py
# Reusable functions shared across all project notebooks.
# Any notebook imports these via:
#   import sys
#   sys.path.append(f'{BASE}/src')
#   from utils import load_wbl, get_year_cols

utils_lines = [
    '"""',
    'utils.py - WBL Trajectory Analysis',
    'Reusable functions imported by all project notebooks.',
    '"""',
    '',
    'import pandas as pd',
    'import os',
    '',
    '',
    'def get_base():',
    '    """Return the project base path on Google Drive."""',
    '    return "/content/drive/MyDrive/wbl_project"',
    '',
    '',
    'def get_year_cols(df, start=1971):',
    '    """',
    '    Return year columns from a WBL dataframe.',
    '    Columns are labelled YR1971, YR1972 etc.',
    '    Only returns years >= start.',
    '    """',
    '    return [c for c in df.columns',
    '            if c.startswith("YR") and int(c[2:]) >= start]',
    '',
    '',
    'def load_wbl(base=None):',
    '    """',
    '    Load the cleaned WBL panel from data/processed/wbl_clean.csv.',
    '    Returns df (full) and df_scores (year columns only).',
    '    """',
    '    if base is None:',
    '        base = get_base()',
    '    path = f"{base}/data/processed/wbl_clean.csv"',
    '    df = pd.read_csv(path, index_col=0)',
    '    year_cols = get_year_cols(df)',
    '    df_scores = df[year_cols].copy()',
    '    return df, df_scores',
    '',
    '',
    'def load_wbl_final(base=None):',
    '    """',
    '    Load the final analysis-ready WBL panel (189 economies, 53 years).',
    '    """',
    '    if base is None:',
    '        base = get_base()',
    '    path = f"{base}/data/processed/wbl_panel_final.csv"',
    '    df = pd.read_csv(path, index_col=0)',
    '    year_cols = get_year_cols(df)',
    '    df_scores = df[year_cols].copy()',
    '    return df, df_scores',
]

utils_content = '\n'.join(utils_lines)

utils_path = f'{BASE}/src/utils.py'
with open(utils_path, 'w') as f:
    f.write(utils_content)

print(f'✅ Created: {utils_path}')

✅ Created: /content/drive/MyDrive/wbl_project/src/utils.py


In [6]:
# Download WBL historical panel
# Indicator: SG.LAW.INDX — WBL overall index
# Source: World Bank Open Data API
# https://data.worldbank.org/indicator/SG.LAW.INDX

RAW_PATH = f'{BASE}/data/raw/wbl_raw.csv'

# Save the raw data
df_raw = wb.data.DataFrame('SG.LAW.INDX',labels=True)
df_raw.to_csv(RAW_PATH)

APIError: APIError: [504] Gateway Time-out (https://api.worldbank.org/v2/en/sources/2/series/SG.LAW.INDX/country/all/time/all?per_page=1000&page=2&format=json)

In [7]:
# Filter aggregates, keep real economies
# The World Bank API returns 266 rows: 217 real economies + 49 aggregate groupings
# We use the World Bank's own aggregate flag to filter.
# Source: World Bank Country and Lending Groups
# https://datahelpdesk.worldbank.org/knowledgebase/articles/906519

all_economies  = list(wb.economy.list())
real_countries = [e['id'] for e in all_economies
                  if not e.get('aggregate', False)]
aggregates = [e['id'] for e in all_economies
                  if e.get('aggregate', False)]

print(f'Real economies: {len(real_countries)}')
print(f'Aggregates:{len(aggregates)}')

df_raw = pd.read_csv(f'{BASE}/data/raw/wbl_raw.csv', index_col=0)
df_countries = df_raw[df_raw.index.isin(real_countries)].copy()

year_cols = [c for c in df_countries.columns
             if c.startswith('YR') and int(c[2:]) >= 1971]
df_clean = df_countries[['Country'] + year_cols].copy()

print(f'\nAfter filtering:')
print(f' Economies: {df_clean.shape[0]}')
print(f'Years: {df_clean.shape[1] - 1} ({year_cols[0]} to {year_cols[-1]})')

# Save the cleaned dataset
CLEAN_PATH = f'{BASE}/data/processed/wbl_clean.csv'
df_clean.to_csv(CLEAN_PATH)
print(f'Saved to: {CLEAN_PATH}')

Real economies: 217
Aggregates:     49

After filtering:
  Economies : 217
  Years     : 55 (YR1971 to YR2025)

✅ Saved to: /content/drive/MyDrive/wbl_project/data/processed/wbl_clean.csv


In [8]:
# Add the latest decisions to the decision.md

dd_path = f'{BASE}/reports/DESIGN_DECISIONS.md'

content = """# DESIGN_DECISIONS.md

---
## DD-000c — Differentiation from Hernández-Lara et al. (2025)

**What they did:**
Hernández-Lara et al. (2025) perform longitudinal analysis of WBL scores
grouped by income level, region, and performance tier. They describe how
trajectories differ across these predefined categories and identify leading
vs lagging economies within those groups.

**What they did not do:**
They do not apply an unsupervised distance-based clustering algorithm to the
raw WBL time series. Their groupings are imposed using existing World Bank
classifications, not discovered from the data. They do not use DTW, do not
compute a trajectory distance matrix, and do not apply SHAP to explain
cluster membership.

**How this project differs:**
This project uses Dynamic Time Warping over the full 1971–2024 WBL panel to
algorithmically identify economies that share reform trajectories regardless
of their income group or region. Cluster membership is then explained using
SHAP applied to a trained classifier — making the explanatory layer
data-driven rather than descriptive.

---

## DD-001 — Filtering aggregates from the WBL panel

**Decision:** Removed World Bank aggregate rows, kept only real economies.
**Method:** Used wbgapi aggregate flag (e.get('aggregate', False)).

**Result before filtering:** 266 rows (217 economies + 49 aggregates)
**Result after filtering:**  217 economies, 55 years (YR1971-YR2025)
**Saved to:** data/processed/wbl_clean.csv

---

"""

with open(dd_path, 'w') as f:
    f.write(content)

print(f'✅ DESIGN_DECISIONS.md initialised at: {dd_path}')

✅ DESIGN_DECISIONS.md initialised at: /content/drive/MyDrive/wbl_project/reports/DESIGN_DECISIONS.md


---
## External Data Collection
### Grounded in Al-Marhubi (2026) Section 2.2

Al-Marhubi identifies the following convergence mechanisms, each of which
maps to a measurable feature:

| Mechanism (Section 2.2) | Feature | Source |
|---|---|---|
| Economic development, access to IFI conditionalities | GDP per capita | WDI NY.GDP.PCAP.CD |
| Women's labour market participation | FLFP % | WDI SL.TLF.CACT.FE.ZS |
| Women's political representation | Women in parliament % | WDI SG.GEN.PARL.ZS |
| Globalization — trade linkages | Trade openness % GDP | WDI NE.TRD.GNFS.ZS |
| Income level — bureaucratic capacity, political will | Income group | WDI metadata |
| Regional peer diffusion | Region | WDI metadata |
| CEDAW — international legal commitment | CEDAW ratification year | UN Treaty Collection |
| Democratisation | Democracy score | V-Dem or Polity V |
| Gender-sensitive aid conditionality | Gender-marked ODA share | OECD DAC CRS |

**Rule:** Raw data only in this notebook. No transformation, no merging.
Each source is saved as a separate file in `data/external/`.
Visualisation and cleaning happen in `data_exploration.ipynb`.

In [9]:
# Collect WDI external indicators (raw)
# Pulls four indicators for all real economies, full time range.
# Saves each as a separate raw file
# Indicators grounded in Al-Marhubi (2026) Section 2.2:
#   NY.GDP.PCAP.CD  — economic development / IFI access
#   SL.TLF.CACT.FE.ZS — female labour force participation
#   SG.GEN.PARL.ZS  — women's political representation
#   NE.TRD.GNFS.ZS  — trade openness (globalisation proxy)


EXTERNAL_INDICATORS = {
    'NY.GDP.PCAP.CD'   : 'gdp_pc',
    'SL.TLF.CACT.FE.ZS': 'flfp',
    'SG.GEN.PARL.ZS'   : 'women_parliament',
    'NE.TRD.GNFS.ZS'   : 'trade_openness',
}

for code, name in EXTERNAL_INDICATORS.items():
    print(f'Pulling {code} ({name})...', end=' ', flush=True)
    d = wb.data.DataFrame(code, economy=real_countries, time=range(1971, 2025))
    d.index.name = 'economy'
    d = d.reset_index()
    out = f'{BASE}/data/external/{name}_raw.csv'
    d.to_csv(out, index=False)
    n_obs = d.notna().sum().sum() - len(d)  # subtract economy col
    print(f'{d.shape[0]} economies | saved to data/external/{name}_raw.csv')


Pulling NY.GDP.PCAP.CD (gdp_pc)... ✅  217 economies | saved to data/external/gdp_pc_raw.csv
Pulling SL.TLF.CACT.FE.ZS (flfp)... 

APIError: APIError: [524]  (https://api.worldbank.org/v2/en/sources/2/series/SL.TLF.CACT.FE.ZS/country/ABW;AFG;AGO;ALB;AND;ARE;ARG;ARM;ASM;ATG;AUS;AUT;AZE;BDI;BEL;BEN;BFA;BGD;BGR;BHR;BHS;BIH;BLR;BLZ;BMU;BOL;BRA;BRB;BRN;BTN;BWA;CAF;CAN;CHE;CHI;CHL;CHN;CIV;CMR;COD;COG;COL;COM;CPV;CRI;CUB;CUW;CYM;CYP;CZE;DEU;DJI;DMA;DNK;DOM;DZA;ECU;EGY;ERI;ESP;EST;ETH;FIN;FJI;FRA;FRO;FSM;GAB;GBR;GEO;GHA;GIB;GIN;GMB;GNB;GNQ;GRC;GRD;GRL;GTM;GUM;GUY;HKG;HND;HRV;HTI;HUN;IDN;IMN;IND;IRL;IRN;IRQ;ISL;ISR;ITA;JAM;JOR;JPN;KAZ;KEN;KGZ;KHM;KIR;KNA;KOR;KWT;LAO;LBN;LBR;LBY;LCA;LIE;LKA;LSO;LTU;LUX;LVA;MAC;MAF;MAR;MCO;MDA;MDG;MDV;MEX;MHL;MKD;MLI;MLT;MMR;MNE;MNG;MNP;MOZ;MRT;MUS;MWI;MYS;NAM;NCL;NER;NGA;NIC;NLD;NOR;NPL;NRU;NZL;OMN;PAK;PAN;PER;PHL;PLW;PNG;POL;PRI;PRK;PRT;PRY;PSE;PYF;QAT;ROU;RUS;RWA;SAU;SDN;SEN;SGP;SLB;SLE;SLV;SMR;SOM;SRB;SSD;STP;SUR;SVK;SVN;SWE;SWZ;SXM;SYC;SYR;TCA;TCD;TGO;THA;TJK;TKM;TLS;TON;TTO;TUN;TUR;TUV;TZA;UGA;UKR;URY;USA;UZB;VCT;VEN;VGB;VIR;VNM;VUT;WSM;XKX;YEM;ZAF;ZMB;ZWE/time/YR1971;YR1972;YR1973;YR1

In [ ]:
# Collect WDI economy metadata (income group, region)
# Both were mentioned inside Al-Marhubi (2026) under section 2.2

meta = wb.economy.DataFrame()[['region', 'incomeLevel', 'name']]
meta.index.name = 'economy'
meta = meta.reset_index()
meta = meta[meta['economy'].isin(real_countries)].copy()

out = f'{BASE}/data/external/wdi_metadata_raw.csv'
meta.to_csv(out, index=False)

print(f'Metadata: {meta.shape[0]} economies')
print(f'Income groups: {meta["incomeLevel"].nunique()} — {sorted(meta["incomeLevel"].dropna().unique())}')
print(f'Regions: {meta["region"].nunique()} — {sorted(meta["region"].dropna().unique())}')
print(f'\n✅ Saved to: data/external/wdi_metadata_raw.csv')

In [ ]:
# Decided to remove CEDAW

dd_path = f'{BASE}/reports/DESIGN_DECISIONS.md'

dd_entry = """
## DD-004e — CEDAW Ratification Year: Excluded

**Decision:** Excluded from Matrix B.

**Empirical basis for relevance:** Al-Marhubi (2026) Section 2.2 explicitly
names CEDAW as an international legal framework consolidating gender equality
as a global policy norm.

**Reason behind the decision:** 13 of 189 panel economies cannot be assigned a
ratification year due to structural reasons: non-UN-member territories
(PSE, HKG, PRI), limited-recognition entities (XKX), and lookup gaps.
Missingness is MNAR — it correlates with political status, not data
availability. Imputation is not defensible.

**Limitation:** CEDAW ratification year is excluded due to
structural coverage gaps for 13 economies (territories and limited-recognition
states).

---
"""

with open(dd_path, 'a') as f:
    f.write(dd_entry)

print('✅ DD-004e recorded: CEDAW excluded with documented justification.')

In [ ]:
# Downlaod Democracy score: Polity V (Political liberalization incentivizes governments to
# appeal to women voters and bolster international legitimacy)
# Also a suggestion presented in Al-Marhubi (2026).
# Source: Polity V (p5v2018.xls)
# AI works

polity_path = f'{BASE}/data/external/polity5_raw.xls'

if not os.path.exists(polity_path):
    print('No democracy file found.')
    print('Download: https://www.systemicpeace.org/inscr/p5v2018.xls')
    print('Upload to: data/external/polity5_raw.xls')

else:
    polity = pd.read_excel(polity_path, usecols=['scode', 'country', 'year', 'polity2'])

    flagged = polity[polity['polity2'].isin([-66, -77, -88])]
    print(f'Special codes removed (-66/-77/-88): {len(flagged)} rows')
    polity  = polity[~polity['polity2'].isin([-66, -77, -88])].copy()

    # ── Full crosswalk: Polity V scode → ISO3 ───────────────────────────────
    POLITY_TO_ISO3 = {
        # Original entries
        'AFG':'AFG','ALB':'ALB','ALG':'DZA','ANG':'AGO','ARG':'ARG',
        'ARM':'ARM','AUL':'AUS','AUS':'AUT','AZE':'AZE','BAH':'BHR',
        'BAN':'BGD','BAR':'BRB','BEL':'BEL','BEN':'BEN','BFO':'BFA',
        'BHU':'BTN','BLR':'BLR','BLZ':'BLZ','BOL':'BOL','BOS':'BIH',
        'BOT':'BWA','BRA':'BRA','BRN':'BRN','BUI':'BDI','CAM':'KHM',
        'CAN':'CAN','CAO':'CMR','CAP':'CPV','CEN':'CAF','CHA':'TCD',
        'CHI':'CHL','CHN':'CHN','COL':'COL','COM':'COM','CON':'COG',
        'COS':'CRI','CRO':'HRV','CUB':'CUB','CYP':'CYP','CZR':'CZE',
        'DEN':'DNK','DJI':'DJI','DOM':'DOM','DRC':'COD','ECU':'ECU',
        'EGY':'EGY','ERI':'ERI','EST':'EST','ETH':'ETH','FIJ':'FJI',
        'FIN':'FIN','FRN':'FRA','GAB':'GAB','GAM':'GMB','GEO':'GEO',
        'GER':'DEU','GHA':'GHA','GRC':'GRC','GUA':'GTM','GUI':'GIN',
        'GUY':'GUY','HAI':'HTI','HND':'HND','HON':'HND','HUN':'HUN',
        'ICE':'ISL','IND':'IND','INS':'IDN','IRE':'IRL','IRN':'IRN',
        'IRQ':'IRQ','ISR':'ISR','ITA':'ITA','JAM':'JAM','JOR':'JOR',
        'JPN':'JPN','KAZ':'KAZ','KEN':'KEN','KGZ':'KGZ','KUW':'KWT',
        'LAO':'LAO','LAT':'LVA','LEB':'LBN','LES':'LSO','LIB':'LBY',
        'LIT':'LTU','LUX':'LUX','MAA':'MRT','MAC':'MKD','MAD':'MDG',
        'MAL':'MYS','MAW':'MWI','MEX':'MEX','MLD':'MDA','MLI':'MLI',
        'MNG':'MNG','MON':'MNE','MOR':'MAR','MOZ':'MOZ','MYA':'MMR',
        'NAM':'NAM','NEP':'NPL','NET':'NLD','NEW':'NZL','NIC':'NIC',
        'NIG':'NER','NIR':'NGA','NOR':'NOR','OMA':'OMN','PAK':'PAK',
        'PAN':'PAN','PAR':'PRY','PER':'PER','PHI':'PHL','POL':'POL',
        'POR':'PRT','QAT':'QAT','ROM':'ROU','RUS':'RUS','RWA':'RWA',
        'SAF':'ZAF','SAU':'SAU','SEN':'SEN','SER':'SRB','SIE':'SLE',
        'SIN':'SGP','SLO':'SVN','SLV':'SVK','SOL':'SLB','SOM':'SOM',
        'SPN':'ESP','SRI':'LKA','SSD':'SSD','SUD':'SDN','SUR':'SUR',
        'SWA':'SWZ','SWD':'SWE','SWZ':'CHE','SYR':'SYR','TAJ':'TJK',
        'TAN':'TZA','THI':'THA','TIM':'TLS','TOG':'TGO','TRI':'TTO',
        'TUN':'TUN','TUR':'TUR','TKM':'TKM','UGA':'UGA','UKR':'UKR',
        'UAE':'ARE','URU':'URY','USA':'USA','UZB':'UZB','VEN':'VEN',
        'VIE':'VNM','YEM':'YEM','ZAM':'ZMB','ZIM':'ZWE',
        # ── Additional entries from unmapped scodes ──────────────────────────
        'BUL':'BGR',  # Bulgaria
        'EQG':'GNQ',  # Equatorial Guinea
        'FJI':'FJI',  # Fiji (alternative code)
        'GNB':'GNB',  # Guinea-Bissau
        'IVO':'CIV',  # Ivory Coast (Cote d'Ivoire)
        'KYR':'KGZ',  # Kyrgyzstan (alternative code)
        'KZK':'KAZ',  # Kazakhstan (alternative code)
        'LBR':'LBR',  # Liberia
        'MAS':'MYS',  # Malaysia (alternative code)
        'MNT':'MNE',  # Montenegro
        'MOD':'MDA',  # Moldova (alternative code)
        'MZM':'MOZ',  # Mozambique (alternative code)
        'PNG':'PNG',  # Papua New Guinea
        'ROK':'KOR',  # Republic of Korea (South Korea)
        'RUM':'ROU',  # Romania (alternative code)
        'SDN':'SDN',  # Sudan (alternative code)
        'SSU':'SSD',  # South Sudan
        'UKG':'GBR',  # United Kingdom (alternative code)
        'YGS':'SRB',  # Yugoslavia → Serbia (closest current state)
        'ZAI':'COD',  # Zaire → DR Congo
        # ── Historical/defunct — intentionally omitted ───────────────────────
        # BAD (Baden), BAV (Bavaria), BNG (Bengal), DRV (North Vietnam),
        # ETI (Ethiopia-Eritrea), ETM (East Timor pre-independence),
        # GCL (Gran Colombia), GDR (East Germany), GFR (West Germany),
        # GMY (Germany pre-unification), MAG (Madagascar alt), NTH (Netherlands alt),
        # OFS (Orange Free State), PAP (Papal States), PKS (Pakistan alt),
        # PMA (Parma), PRK (North Korea — not in WBL panel),
        # RVN (South Vietnam), SAL (El Salvador alt), SAR (Sardinia),
        # SAX (Saxony), SIC (Sicily), TAW (Taiwan — not in WBL panel),
        # TAZ (Tanzania alt), TUS (Tuscany), UPC (Upper Canada),
        # USR (USSR), WRT (Wurttemberg), YAR (Yemen Arab Republic),
        # YPR (Yemen PDR), YUG (Yugoslavia — mapped via YGS to SRB),
        # CHL/CZE/KOR — duplicate scodes handled above
    }

    polity['economy'] = polity['scode'].map(POLITY_TO_ISO3)

    n_mapped   = polity['economy'].notna().sum()
    n_unmapped = polity['economy'].isna().sum()
    print(f'\nPolity V loaded: {polity.shape[0]} rows')
    print(f'  Mapped to ISO3:   {polity["economy"].nunique()} unique economies')
    print(f'  Unmapped rows:    {n_unmapped} (historical/defunct states — expected)')

    # Check coverage against WBL panel
    import pandas as pd
    panel     = pd.read_csv(f'{BASE}/data/processed/wbl_panel_final.csv', index_col=0)
    ECONOMIES = panel.index.tolist()

    dem_1971   = polity[polity['year'] == 1971]
    matched    = dem_1971[dem_1971['economy'].isin(ECONOMIES)]
    not_matched = [e for e in ECONOMIES if e not in dem_1971['economy'].values]

    print(f'\n  Panel economies with 1971 value: {len(matched)} / {len(ECONOMIES)}')
    print(f'  Missing at 1971 (will use earliest available): {len(not_matched)}')
    if not_matched:
        print(f'  {not_matched[:15]}')

    # Save
    polity.to_csv(f'{BASE}/data/external/polity5_raw.csv', index=False)
    print(f'\n Saved: data/external/polity5_raw.csv')
    print(f'Years: {polity["year"].min()} — {polity["year"].max()}')
    print(f'Polity2 range: {polity["polity2"].min()} to {polity["polity2"].max()}')

In [ ]:
# Gender-marked ODA: EXCLUDED (DD-004)
# Al-Marhubi (2026) Section 2.2 names gender-sensitive aid conditionality
# as a convergence driver. Excluded due to data quality constraints:
# the OECD DAC CRS gender marker was not applied systematically before 2002,
# creating sparse coverage for the earliest and most formative reform years.

dd_path = f'{BASE}/reports/DESIGN_DECISIONS.md'

dd_entry = """
## DD-004 — Gender-Marked ODA: Excluded

**Decision:** Excluded from Matrix B.
**Explination:** Al-Marhubi (2026) Section 2.2 explicitly
names gender-sensitive aid conditionality as a convergence driver. Two
independent panel studies found it predicts subsequent WBL reforms.

**Reason:** The OECD DAC CRS gender marker was not applied
systematically before 2002. For a panel starting in 1971, coverage would
be sparse for the earliest and most formative years of reform trajectories,
introducing systematic bias toward high-income recipient economies in the
post-2002 period. Gender-marked ODA share is excluded due to
data availability constraints in the pre-2002 period. Future work with
post-2002 panels may benefit from including this variable.

---
"""

with open(dd_path, 'a') as f:
    f.write(dd_entry)

In [ ]:
# External collection summary [AI]

panel = pd.read_csv(f'{BASE}/data/processed/wbl_panel_final.csv', index_col=0)
ECONOMIES = panel.index.tolist()

print('EXTERNAL DATA COLLECTION SUMMARY')

checks = [
    ('GDP per capita', f'{BASE}/data/external/gdp_pc_raw.csv'),
    ('FLFP', f'{BASE}/data/external/flfp_raw.csv'),
    ('Women in parliament',  f'{BASE}/data/external/women_parliament_raw.csv'),
    ('Trade openness', f'{BASE}/data/external/trade_openness_raw.csv'),
    ('WDI metadata', f'{BASE}/data/external/wdi_metadata_raw.csv'),
    ('Democracy (Polity V)', f'{BASE}/data/external/polity5_raw.csv'),
]

all_present = True
for label, path in checks:
    if os.path.exists(path):
        print(f' {label}')
    else:
        print(f' {label} — MISSING')
        all_present = False

print(f'CEDAW: excluded — DD-004e recorded')
print(f'  ODA:   excluded — DD-004  recorded ')

print('FEATURE LIST (Matrix B)')
features = [
    ('GDP per capita (log)',   'WDI NY.GDP.PCAP.CD',        'Initial condition 1971'),
    ('FLFP %',                 'WDI SL.TLF.CACT.FE.ZS',     'Initial condition 1971'),
    ('Women in parliament %',  'WDI SG.GEN.PARL.ZS',        'Initial condition 1971'),
    ('Trade openness % GDP',   'WDI NE.TRD.GNFS.ZS',        'Initial condition 1971'),
    ('Income group',           'WDI metadata',               'Ordinal 1–4'),
    ('Region',                 'WDI metadata',               'Categorical'),
    ('Democracy score',        'Polity V polity2',           'Initial condition 1971'),
]
for name, source, note in features:
    print(f'  {name:<28} {source:<30} {note}')

print('\n=== STATUS ===\n')
if all_present:
    print(' All files present — 7 features confirmed — proceed to data_exploration.ipynb')
else:
    print(' Files missing — resolve above before data_exploration.ipynb')